# 第二章：Conda 与 Python 包管理

## 环境隔离、依赖管理、可复现的 AI 开发工作流

AI 项目依赖大量第三方库（PyTorch, Transformers, NumPy 等），不同项目可能需要**不同版本**的同一库。
本章介绍 Python 环境管理的标准工具链，确保每个项目的依赖**独立、可复现、不冲突**。


## 2.1 为什么需要虚拟环境？

### 问题场景

假设你同时维护两个项目：

| 项目 | 需要 PyTorch | 需要 NumPy |
|------|-------------|------------|
| 项目 A（旧） | 1.13.0 | 1.23.0 |
| 项目 B（新） | 2.5.0 | 2.0.0 |

如果全局安装，PyTorch 和 NumPy 只能有一个版本——必然有一个项目无法运行。

### 核心概念

```
全局 Python (系统级)
├── venv_project_a/     ← 独立的 Python + site-packages
│   ├── torch==1.13.0
│   └── numpy==1.23.0
├── venv_project_b/     ← 完全隔离，互不影响
│   ├── torch==2.5.0
│   └── numpy==2.0.0
└── conda_env_dl/       ← Conda 环境，可装非 Python 依赖 (CUDA, C++ libs)
    ├── pytorch=2.5.0
    └── cudatoolkit=12.1
```

### 三类主流方案

| 工具 | 适用场景 | 特点 |
|------|---------|------|
| `venv` | 纯 Python 项目 | Python 内置，轻量，仅管理 Python 包 |
| `virtualenv` | 需要 Python 版本切换 | 比 venv 功能更丰富 |
| `conda` | AI/科学计算 | 管理 Python + 非 Python 依赖（CUDA, MKL, C++ libs）|


## 2.2 venv — Python 内置虚拟环境

`venv` 从 Python 3.3 开始内置，无需安装。创建虚拟环境会复制一个**轻量级 Python 解释器**和独立的 `site-packages` 目录。


In [ ]:
# === 在终端中执行（非 notebook cell）===
# 
# # 1. 创建虚拟环境
# python3 -m venv myenv
#
# # 2. 激活环境
# # Linux/macOS:
# source myenv/bin/activate
# # Windows:
# myenv\Scripts\activate
#
# # 3. 确认环境
# which python    # 应指向 myenv/bin/python
# pip list        # 应为空（仅 pip 和 setuptools）
#
# # 4. 安装依赖
# pip install numpy pandas matplotlib
#
# # 5. 导出依赖列表
# pip freeze > requirements.txt
#
# # 6. 退出环境
# deactivate


In [ ]:
# === 在 Python 中检查当前环境 ===
import sys
import os

print(f"Python 路径: {sys.executable}")
print(f"是否虚拟环境: {sys.prefix != sys.base_prefix}")
print(f"site-packages: {sys.path[-3:]}")

# 查看已安装包的版本
import importlib.metadata as im

packages_to_check = ["numpy", "pandas", "matplotlib"]
for pkg in packages_to_check:
    try:
        version = im.version(pkg)
        print(f"  {pkg}=={version}")
    except im.PackageNotFoundError:
        print(f"  {pkg}: 未安装")


## 2.3 pip — Python 包安装器

`pip` 是 Python 官方包管理器，从 **PyPI** (Python Package Index) 下载安装包。


In [ ]:
# === pip 常用命令（在终端执行）===
#
# # 安装单个包
# pip install numpy
#
# # 安装特定版本
# pip install torch==2.0.0
#
# # 同时安装多个包
# pip install numpy pandas scikit-learn
#
# # 从 requirements.txt 批量安装
# pip install -r requirements.txt
#
# # 升级包
# pip install --upgrade torch
#
# # 卸载包
# pip uninstall numpy
#
# # 查看已安装包
# pip list
# pip show torch    # 详细信息
#
# # 搜索包（注意：PyPI 搜索 API 可能变更）
# pip search transformers  # 部分版本已禁用


### pip 镜像源

国内使用 PyPI 官方源速度较慢，建议配置镜像：

```bash
# 临时使用
pip install torch -i https://pypi.tuna.tsinghua.edu.cn/simple

# 永久配置
pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple
```

**常用国内镜像：**
- 清华：`https://pypi.tuna.tsinghua.edu.cn/simple`
- 阿里云：`https://mirrors.aliyun.com/pypi/simple/`
- 中科大：`https://pypi.mirrors.ustc.edu.cn/simple/`


## 2.4 Conda — 数据科学/AI 的首选环境管理器

### Conda vs pip

| | pip | conda |
|---|-----|-------|
| 安装内容 | 仅 Python 包 | Python 包 + C/C++/R 库 |
| 依赖解析 | 较简单（可能冲突） | SAT solver，更严格 |
| 环境管理 | 需配合 venv | 内置环境管理 |
| 包来源 | PyPI | Anaconda Cloud / conda-forge |
| CUDA 管理 | 手动安装 | `cudatoolkit` 一键安装 |
| 典型场景 | Web 开发、脚本 | 数据科学、深度学习、生物信息 |


In [ ]:
# === Conda 常用命令（在终端执行）===
#
# # ---- 环境管理 ----
# conda create -n myenv python=3.11        # 创建环境
# conda activate myenv                     # 激活环境
# conda deactivate                         # 退出环境
# conda env list                           # 列出所有环境
# conda remove -n myenv --all              # 删除环境
# conda env export > environment.yml       # 导出环境
# conda env create -f environment.yml      # 从文件创建环境
#
# # ---- 包管理 ----
# conda install numpy pandas               # 安装包
# conda install pytorch torchvision \
#   pytorch-cuda=12.1 -c pytorch -c nvidia # 带 channel 安装
# conda list                               # 列出已安装包
# conda update --all                       # 更新所有包
# conda remove numpy                       # 卸载包
#
# # ---- 配置 ----
# conda config --show channels             # 查看 channel
# conda config --add channels conda-forge  # 添加 channel


### Conda channel 与优先级

Conda 从 **channel**（软件仓库）下载包。默认 channel 是 Anaconda 官方仓库。

**推荐 channel 配置顺序：**
```
conda-forge > pytorch > nvidia > defaults
```

`conda-forge` 是社区维护的最大 channel，包更新更快、更全。

### Miniconda vs Anaconda

| | Miniconda | Anaconda |
|---|----------|----------|
| 下载大小 | ~50 MB | ~500 MB |
| 预装包 | 仅 conda + python | 150+ 数据科学包 |
| 适用场景 | 需要灵活控制依赖 | 快速上手、教学 |
| **推荐** | ✅ | 仅新手 |

AI 开发**强烈推荐 Miniconda**：按需安装，避免依赖冲突。


## 2.5 requirements.txt — 精确锁定依赖版本

`requirements.txt` 是 Python 项目的**依赖清单**，确保任何人在任意机器上可以通过 `pip install -r requirements.txt` 复现完全相同的环境。


In [ ]:
# === 生成 requirements.txt ===
# 
# 方式 1：导出当前环境所有包（最常用）
# pip freeze > requirements.txt
#
# 方式 2：使用 pip-tools 精确管理（推荐正式项目）
# pip install pip-tools
# # 编辑 requirements.in（只写直接依赖）
# # numpy
# # torch>=2.0
# # transformers
# pip-compile requirements.in  # 生成锁定的 requirements.txt

# === 示例：一个典型 AI 项目的 requirements.txt ===
sample_requirements = """
# Core
numpy>=1.24,<2.0
scipy>=1.10
pandas>=2.0

# Deep Learning
torch>=2.0
torchvision>=0.15

# NLP
transformers>=4.30
tokenizers>=0.13
datasets>=2.12

# Visualization
matplotlib>=3.7
seaborn>=0.12

# Dev Tools
jupyter>=1.0
ipykernel>=6.0
"""

with open("/tmp/sample_requirements.txt", "w") as f:
    f.write(sample_requirements)

print("示例 requirements.txt:")
print(sample_requirements)
import os; os.remove("/tmp/sample_requirements.txt")


### 版本号规范

```python
torch==2.5.0       # 精确版本（最安全但不够灵活）
torch>=2.0         # 至少 2.0，允许更高
torch>=2.0,<3.0    # 兼容范围（推荐——明确上下界）
torch~=2.5.0       # 兼容版本（>=2.5.0, <2.6.0）
torch!=1.13.0      # 排除特定版本（已知有 bug）
```


## 2.6 environment.yml — Conda 的依赖描述文件

`environment.yml` 比 `requirements.txt` 更强大：可以指定 Python 版本、channel，还可以包含非 Python 依赖（如 CUDA）。


In [ ]:
# === 示例 environment.yml ===
sample_env_yml = """
name: ai-learning
channels:
  - pytorch
  - nvidia
  - conda-forge
  - defaults
dependencies:
  - python=3.11
  - pip
  - cudatoolkit=12.1
  - numpy>=1.24
  - scipy>=1.10
  - pandas>=2.0
  - matplotlib>=3.7
  - jupyter>=1.0
  - pip:
    - torch>=2.0
    - torchvision>=0.15
    - transformers>=4.30
    - datasets>=2.12
    - accelerate>=0.20
"""

print("示例 environment.yml:")
print(sample_env_yml)

with open("/tmp/environment.yml", "w") as f:
    f.write(sample_env_yml)
import os; os.remove("/tmp/environment.yml")

print("\n使用方式:")
print("  conda env create -f environment.yml")
print("  conda activate ai-learning")


## 2.7 AI 项目依赖管理最佳实践

### 推荐项目结构

```
ai-learning/
├── README.md
├── requirements.txt          # pip 用户
├── environment.yml           # conda 用户
├── notebooks/                # Jupyter 笔记
│   ├── ch01_python_basics.ipynb
│   └── ch02_env_management.ipynb
├── src/                      # 源代码
│   ├── __init__.py
│   ├── models.py
│   └── utils.py
└── data/                     # 数据（.gitignore）
    └── .gitkeep
```


In [ ]:
# === 依赖检查脚本 ===
"""
在项目根目录运行此脚本，检查关键依赖是否正确安装。
"""

import sys

def check_package(name, min_version=None):
    try:
        import importlib.metadata as im
        version = im.version(name)
        if min_version:
            from packaging import version as pv
            if pv.parse(version) < pv.parse(min_version):
                return f"⚠ {name}=={version} (require >={min_version})"  # 返回结果
        return f"✅ {name}=={version}"  # 返回结果
    except ImportError:
        return f"❌ {name}: 未安装"  # 返回结果

# 检查列表
deps = [
    ("numpy", "1.24"),
    ("pandas", "2.0"),
    ("matplotlib", "3.7"),
    ("torch", "2.0"),
    ("torchvision", "0.15"),
    ("transformers", "4.30"),
]

print("Python:", sys.version.split()[0])
print("-" * 50)
for name, min_ver in deps:
    print(check_package(name, min_ver))


## 2.8 常见问题排查

### 1. "externally-managed-environment" 错误

Ubuntu 24.04 / Debian 12+ 的 Python 默认禁止 `pip install` 全局安装。

**解决方案：**
```bash
# 方案 A：创建虚拟环境（推荐）
python3 -m venv myenv && source myenv/bin/activate

# 方案 B：使用 pipx（CLI 工具专用）
pipx install jupyter

# 方案 C：强制安装（不推荐）
pip install --break-system-packages numpy
```

### 2. CUDA/cuDNN 版本不匹配

PyTorch 需要与 CUDA 版本匹配。检查方法：
```python
import torch
print(torch.cuda.is_available())  # True 表示 GPU 可用
print(torch.version.cuda)         # PyTorch 编译时的 CUDA 版本
```

### 3. Conda 环境不激活

检查 shell 是否初始化 conda：
```bash
conda init bash   # 或 zsh / fish
# 重启终端后再试
```


## 本章小结

| 概念 | 关键命令 |
|------|----------|
| 创建 venv | `python3 -m venv env_name` |
| 激活环境 | `source env_name/bin/activate` (Linux) |
| 安装包 | `pip install package` |
| 导出依赖 | `pip freeze > requirements.txt` |
| 复现环境 | `pip install -r requirements.txt` |
| Conda 创建 | `conda create -n env_name python=3.11` |
| Conda 导出 | `conda env export > environment.yml` |

✅ 第二章完成！你已掌握 AI 开发的环境管理工具链。
